# Día 1.1 — Regularización: Ridge, Lasso y Elastic Net

**Módulo 3: Machine Learning Aplicado — CEC EPN / Banco Central del Ecuador**

Este notebook es una **extensión del Día 1**. No busca reemplazar la regresión lineal, sino mostrar tres variantes que modifican la forma en que el modelo aprende sus coeficientes:

- **Ridge**
- **Lasso**
- **Elastic Net**

La idea principal será:

> **Regresión lineal + penalización de coeficientes = regularización**

Trabajaremos con el mismo dataset didáctico del Día 1 y compararemos todos los modelos usando **el mismo Train y Validation**.

## Objetivos

Al terminar deberías poder explicar:

- por qué un modelo puede beneficiarse de regularización;
- qué significa el hiperparámetro $\alpha$;
- cuál es la diferencia entre penalización L1 y L2;
- por qué debemos escalar las variables antes de Ridge, Lasso y Elastic Net;
- por qué Lasso puede dejar coeficientes exactamente en cero;
- cómo combina Elastic Net las ideas de Ridge y Lasso;
- cómo comparar estos modelos sin utilizar Test para escoger el mejor.

> Este notebook está pensado como una **introducción visual y práctica**. Los árboles de decisión se estudiarán en el Día 2.

# 1. Librerías

Usaremos las mismas herramientas del Día 1 y añadiremos:

- `Ridge`
- `Lasso`
- `ElasticNet`
- `Pipeline`

`Pipeline` nos permitirá encadenar dos pasos:

$$
\text{Escalado} \rightarrow \text{Modelo}
$$

De esta forma, el escalado se aprende únicamente con los datos de entrenamiento.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# 2. Cargar el mismo dataset del Día 1

Utilizaremos nuevamente:

`actividad_economica_demo.csv`

El archivo contiene datos **sintéticos y didácticos**. No representa estadísticas oficiales.

El notebook supone una estructura como:

```text
modulo3_ml/
├── datos/
│   └── actividad_economica_demo.csv
└── notebooks/
    ├── 01_Dia1_Datos_y_Regresion.ipynb
    └── 01_1_Dia1_Regularizacion_Ridge_Lasso_ElasticNet.ipynb
```

In [ ]:
DATA_DIR = Path('../datos')

if not DATA_DIR.exists():
    DATA_DIR = Path('datos')

archivo = DATA_DIR / 'actividad_economica_demo.csv'

if not archivo.exists():
    raise FileNotFoundError(
        'No se encontró actividad_economica_demo.csv. '
        'Coloca el notebook dentro de notebooks/ o junto a la carpeta datos/.'
    )

df = pd.read_csv(archivo)

print('Archivo:', archivo.resolve())
print('Dimensiones:', df.shape)

df.head()

## 2.1 Preparación mínima

Repetimos únicamente las transformaciones necesarias para que este notebook pueda ejecutarse de manera independiente:

1. eliminamos duplicados;
2. retiramos `tipo_dato`, porque es constante;
3. retiramos el outlier didáctico identificado en el Día 1;
4. completamos los valores faltantes numéricos con la mediana;
5. convertimos `region` y `sector` mediante One-Hot Encoding.

> En un flujo de producción, la imputación y otras transformaciones deberían aprenderse únicamente desde Train mediante un pipeline. Aquí mantenemos la misma preparación del Día 1 para concentrarnos en regularización.

In [ ]:
df_limpio = df.drop_duplicates().copy()

if 'tipo_dato' in df_limpio.columns:
    df_limpio = df_limpio.drop(columns=['tipo_dato'])

# Outlier didáctico identificado previamente.
if 33 in df_limpio.index:
    df_limpio = df_limpio.drop(index=33)

columnas_numericas = df_limpio.select_dtypes(include='number').columns

for col in columnas_numericas:
    df_limpio[col] = df_limpio[col].fillna(
        df_limpio[col].median()
    )

df_limpio = df_limpio.reset_index(drop=True)

print('Dimensiones después de limpiar:', df_limpio.shape)
print('Nulos restantes:', int(df_limpio.isna().sum().sum()))

# 3. Preparar X e y

Usaremos las mismas variables del mejor modelo lineal del Día 1.

Variables numéricas:

- inversión;
- precio del petróleo;
- exportaciones;
- confianza;
- inflación;
- desempleo.

También incorporaremos:

- región;
- sector.

La variable objetivo continúa siendo:

$$
y = \text{crecimiento de la actividad económica}
$$

In [ ]:
features_num = [
    'inversion_mill',
    'precio_petroleo_usd',
    'exportaciones_mill',
    'indice_confianza',
    'inflacion_pct',
    'desempleo_pct'
]

X = pd.get_dummies(
    df_limpio[
        features_num + ['region', 'sector']
    ],
    columns=['region', 'sector'],
    drop_first=True,
    dtype=int
)

y = df_limpio['crecimiento_actividad_pct']

print('X:', X.shape)
print('y:', y.shape)
print('\nVariables utilizadas:')
print(X.columns.tolist())

# 4. Train, Validation y Test

Mantendremos el esquema:

- **70 % Train**
- **15 % Validation**
- **15 % Test**

En este notebook utilizaremos **Train para aprender** y **Validation para comparar** Ridge, Lasso y Elastic Net.

No necesitamos mirar Test para entender estos modelos.

Esto mantiene la idea del Día 1:

> **Validation sirve para tomar decisiones. Test se reserva para una evaluación final.**

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)

print('Train:', X_train.shape)
print('Validation:', X_val.shape)
print('Test:', X_test.shape)

# 5. Punto de partida: regresión lineal múltiple

Antes de regularizar necesitamos un punto de comparación.

La regresión lineal múltiple aprende:

$$
\hat y =
b_0 +
b_1x_1 +
b_2x_2 +
\cdots +
b_px_p
$$

y escoge los coeficientes buscando minimizar el error cuadrático.

El problema es que, cuando tenemos muchas variables relacionadas entre sí, ruido o demasiada flexibilidad, algunos coeficientes pueden adquirir magnitudes innecesariamente grandes.

La **regularización** añade una penalización para evitarlo.

In [ ]:
def metricas_regresion(y_real, y_pred):
    mse = mean_squared_error(y_real, y_pred)

    return {
        'MAE': mean_absolute_error(y_real, y_pred),
        'MSE': mse,
        'RMSE': np.sqrt(mse),
        'R2': r2_score(y_real, y_pred),
        'Bias': np.mean(
            np.asarray(y_pred) - np.asarray(y_real)
        )
    }


modelo_lineal = LinearRegression()

modelo_lineal.fit(
    X_train,
    y_train
)

pred_lineal_train = modelo_lineal.predict(X_train)
pred_lineal_val = modelo_lineal.predict(X_val)

pd.DataFrame({
    'Train': metricas_regresion(
        y_train,
        pred_lineal_train
    ),
    'Validation': metricas_regresion(
        y_val,
        pred_lineal_val
    )
}).T.round(3)

# 6. ¿Qué significa regularizar?

Supongamos que tenemos:

$$
\hat y=b_0+b_1x_1+b_2x_2+\cdots+b_px_p
$$

La regresión lineal se concentra principalmente en reducir el error:

$$
\text{Error}
$$

Los modelos regularizados intentan equilibrar dos objetivos:

$$
\text{Error}
+
\text{Penalización por coeficientes grandes}
$$

La idea intuitiva es:

> El modelo debe explicar los datos, pero no debería utilizar coeficientes exageradamente grandes si no son necesarios.

El encargado de controlar la intensidad de la penalización es:

$$
\alpha
$$

- $\alpha$ pequeño → poca regularización;
- $\alpha$ grande → mayor regularización.

Si penalizamos demasiado, el modelo puede quedar excesivamente simplificado y caer en **underfitting**.

# 7. ¿Por qué debemos escalar?

Observa las escalas de nuestras variables:

- inflación puede estar alrededor de `2` o `3`;
- inversión puede estar en cientos;
- exportaciones también puede estar en cientos.

Una penalización actúa directamente sobre los **coeficientes**. Si las variables utilizan escalas muy diferentes, comparar sus coeficientes puede ser injusto.

Por eso utilizaremos:

```python
StandardScaler()
```

que transforma cada variable aproximadamente a:

$$
z=\frac{x-\mu}{\sigma}
$$

donde:

- $\mu$ es la media;
- $\sigma$ es la desviación estándar.

Después del escalado, las variables quedan en una escala comparable.

Usaremos un `Pipeline`:

```text
X
↓
StandardScaler
↓
Ridge / Lasso / Elastic Net
↓
Predicción
```

# 8. Ridge — penalización L2

Ridge agrega una penalización basada en el **cuadrado de los coeficientes**.

Conceptualmente:

$$
\text{Costo Ridge}
=
\text{Error}
+
\alpha
\sum_{j=1}^{p} b_j^2
$$

A esta penalización se la conoce como **L2**.

## ¿Qué intenta hacer Ridge?

Ridge tiende a **reducir** la magnitud de los coeficientes.

Por ejemplo:

```text
coeficiente grande
        ↓
Ridge lo penaliza
        ↓
coeficiente más pequeño
```

Normalmente Ridge **no elimina variables completamente**. Sus coeficientes pueden acercarse a cero, pero suelen mantenerse diferentes de cero.

Es especialmente útil cuando varias variables contienen información parecida o están correlacionadas.

## El parámetro `alpha`

Usaremos inicialmente:

```python
alpha=10
```

Este valor es **didáctico**, no significa que sea universalmente el mejor.

Más adelante veremos que cambiar $\alpha$ cambia la intensidad de la regularización.

In [ ]:
modelo_ridge = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=10))
])

modelo_ridge.fit(
    X_train,
    y_train
)

pred_ridge_train = modelo_ridge.predict(X_train)
pred_ridge_val = modelo_ridge.predict(X_val)

pd.DataFrame({
    'Train': metricas_regresion(
        y_train,
        pred_ridge_train
    ),
    'Validation': metricas_regresion(
        y_val,
        pred_ridge_val
    )
}).T.round(3)

## 8.1 Coeficientes de Ridge

Como utilizamos `StandardScaler`, los coeficientes corresponden a variables **escaladas**.

Esto es útil para observar cuánto los está reduciendo la penalización, pero ya no debemos interpretar directamente un coeficiente como:

> “por cada millón adicional...”

porque el modelo está trabajando internamente con desviaciones estándar.

In [ ]:
coef_ridge = pd.Series(
    modelo_ridge.named_steps['model'].coef_,
    index=X.columns,
    name='Ridge'
)

coef_ridge.sort_values(
    key=np.abs,
    ascending=False
).round(4)

## 8.2 ¿Qué ocurre si aumentamos `alpha`?

Vamos a probar diferentes valores manteniendo exactamente el mismo Train y Validation.

No estamos haciendo validación cruzada todavía. Solo queremos **observar el efecto del hiperparámetro**.

In [ ]:
alphas_ridge = [0.01, 0.1, 1, 10, 30, 100]

resultados_ridge = []

for alpha in alphas_ridge:

    modelo = Pipeline([
        ('scaler', StandardScaler()),
        ('model', Ridge(alpha=alpha))
    ])

    modelo.fit(X_train, y_train)

    pred = modelo.predict(X_val)

    resultados_ridge.append({
        'alpha': alpha,
        **metricas_regresion(y_val, pred)
    })

pd.DataFrame(resultados_ridge).round(3)

Observa que **un `alpha` mayor no garantiza un mejor modelo**.

Existe un equilibrio:

```text
muy poca regularización
        ↕
regularización adecuada
        ↕
demasiada regularización
```

Por eso `alpha` es un **hiperparámetro**: no lo aprende directamente la regresión durante `fit()`. Nosotros debemos escogerlo mediante un proceso de validación.

# 9. Lasso — penalización L1

Lasso utiliza el **valor absoluto de los coeficientes**:

$$
\text{Costo Lasso}
=
\text{Error}
+
\alpha
\sum_{j=1}^{p}|b_j|
$$

Esta penalización se denomina **L1**.

Su característica más interesante es que puede llevar algunos coeficientes exactamente a:

$$
b_j=0
$$

Cuando eso ocurre, esa variable deja de contribuir a la predicción.

Por esta razón Lasso también puede funcionar como una forma de **selección automática de variables**.

Usaremos:

```python
alpha=0.03
```

únicamente como valor demostrativo.

In [ ]:
modelo_lasso = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Lasso(
        alpha=0.03,
        max_iter=20000
    ))
])

modelo_lasso.fit(
    X_train,
    y_train
)

pred_lasso_train = modelo_lasso.predict(X_train)
pred_lasso_val = modelo_lasso.predict(X_val)

pd.DataFrame({
    'Train': metricas_regresion(
        y_train,
        pred_lasso_train
    ),
    'Validation': metricas_regresion(
        y_val,
        pred_lasso_val
    )
}).T.round(3)

## 9.1 ¿Qué variables dejó en cero?

Extraemos los coeficientes del modelo.

Recuerda nuevamente que están expresados sobre variables escaladas.

In [ ]:
coef_lasso = pd.Series(
    modelo_lasso.named_steps['model'].coef_,
    index=X.columns,
    name='Lasso'
)

tabla_lasso = pd.DataFrame({
    'coeficiente': coef_lasso,
    'eliminada_por_lasso': np.isclose(
        coef_lasso,
        0
    )
})

tabla_lasso.round(4)

In [ ]:
cantidad_cero = np.isclose(
    coef_lasso,
    0
).sum()

print(
    'Coeficientes exactamente en cero:',
    cantidad_cero,
    'de',
    len(coef_lasso)
)

Si Lasso coloca un coeficiente en cero, no significa automáticamente que la variable sea **económicamente inútil** o que no tenga relación causal.

Significa únicamente que, para **este dataset, este objetivo, este conjunto de variables y este valor de $\alpha$**, el modelo pudo prescindir de ella para construir su predicción.

## 9.2 Efecto de `alpha` en Lasso

Con Lasso el efecto de aumentar $\alpha$ suele ser más visible porque cada vez más coeficientes pueden convertirse exactamente en cero.

In [ ]:
alphas_lasso = [0.001, 0.01, 0.03, 0.05, 0.1, 0.3]

resultados_lasso = []

for alpha in alphas_lasso:

    modelo = Pipeline([
        ('scaler', StandardScaler()),
        ('model', Lasso(
            alpha=alpha,
            max_iter=20000
        ))
    ])

    modelo.fit(X_train, y_train)

    pred = modelo.predict(X_val)

    coeficientes = modelo.named_steps['model'].coef_

    resultados_lasso.append({
        'alpha': alpha,
        'coeficientes_en_cero':
            int(np.isclose(
                coeficientes,
                0
            ).sum()),
        **metricas_regresion(y_val, pred)
    })

pd.DataFrame(resultados_lasso).round(3)

Aquí podemos observar dos cosas simultáneamente:

1. cómo cambian las métricas;
2. cuántas variables son eliminadas.

Un modelo con más coeficientes en cero **no es necesariamente mejor**. La decisión debe basarse en el desempeño y en el objetivo del problema.

# 10. Elastic Net — combinación de L1 y L2

Elastic Net combina las ideas de Ridge y Lasso.

Conceptualmente:

$$
\text{Costo}
=
\text{Error}
+
\alpha
\left[
\rho
\sum |b_j|
+
(1-\rho)
\sum b_j^2
\right]
$$

En Scikit-learn, $\rho$ se controla con:

```python
l1_ratio
```

Por ejemplo:

```text
l1_ratio cercano a 1
→ comportamiento más parecido a Lasso

l1_ratio cercano a 0
→ comportamiento más parecido a Ridge
```

Usaremos:

```python
alpha=0.05
l1_ratio=0.5
```

para dar el mismo peso conceptual a ambos tipos de penalización.

> Estos valores son demostrativos, no una optimización definitiva.

In [ ]:
modelo_elastic = Pipeline([
    ('scaler', StandardScaler()),
    ('model', ElasticNet(
        alpha=0.05,
        l1_ratio=0.5,
        max_iter=20000
    ))
])

modelo_elastic.fit(
    X_train,
    y_train
)

pred_elastic_train = modelo_elastic.predict(X_train)
pred_elastic_val = modelo_elastic.predict(X_val)

pd.DataFrame({
    'Train': metricas_regresion(
        y_train,
        pred_elastic_train
    ),
    'Validation': metricas_regresion(
        y_val,
        pred_elastic_val
    )
}).T.round(3)

In [ ]:
coef_elastic = pd.Series(
    modelo_elastic.named_steps['model'].coef_,
    index=X.columns,
    name='Elastic Net'
)

coef_elastic.sort_values(
    key=np.abs,
    ascending=False
).round(4)

# 11. Comparación de los cuatro modelos

Ahora compararemos:

1. regresión lineal múltiple;
2. Ridge;
3. Lasso;
4. Elastic Net.

Todos utilizan:

- las mismas observaciones de Train;
- las mismas observaciones de Validation;
- las mismas variables.

Por tanto, la comparación es justa.

Recordemos:

- **MAE menor** → mejor;
- **MSE menor** → mejor;
- **RMSE menor** → mejor;
- **R² mayor** → mejor;
- **Bias cercano a cero** → menor tendencia sistemática a sobreestimar o subestimar.

In [ ]:
comparacion = pd.DataFrame([
    {
        'modelo': 'Lineal múltiple',
        **metricas_regresion(
            y_val,
            pred_lineal_val
        )
    },
    {
        'modelo': 'Ridge',
        **metricas_regresion(
            y_val,
            pred_ridge_val
        )
    },
    {
        'modelo': 'Lasso',
        **metricas_regresion(
            y_val,
            pred_lasso_val
        )
    },
    {
        'modelo': 'Elastic Net',
        **metricas_regresion(
            y_val,
            pred_elastic_val
        )
    }
])

comparacion.round(3)

## ¿Qué debemos concluir?

No debemos esperar que Ridge, Lasso o Elastic Net siempre superen a la regresión lineal.

La regularización es especialmente útil cuando:

- tenemos muchas variables;
- existen variables correlacionadas;
- algunos coeficientes se vuelven inestables;
- queremos reducir complejidad;
- queremos disminuir overfitting.

En un dataset relativamente pequeño y controlado, las diferencias pueden ser reducidas.

El objetivo de esta práctica no es declarar que uno de estos tres algoritmos es siempre superior, sino comprender **cómo modifican el aprendizaje de los coeficientes**.

# 12. Comparación de coeficientes

Para comparar correctamente las magnitudes, entrenaremos también una regresión lineal sobre datos escalados.

Así todos los coeficientes de la siguiente tabla estarán expresados sobre la misma escala.

In [ ]:
modelo_lineal_escalado = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

modelo_lineal_escalado.fit(
    X_train,
    y_train
)

coef_lineal_escalado = pd.Series(
    modelo_lineal_escalado.named_steps['model'].coef_,
    index=X.columns,
    name='Lineal'
)

comparacion_coef = pd.concat(
    [
        coef_lineal_escalado,
        coef_ridge,
        coef_lasso,
        coef_elastic
    ],
    axis=1
)

comparacion_coef.round(4)

Ahora podemos observar directamente el efecto de la regularización:

- **Lineal:** no penaliza coeficientes;
- **Ridge:** tiende a reducirlos;
- **Lasso:** puede reducirlos hasta exactamente cero;
- **Elastic Net:** combina ambos comportamientos.

In [ ]:
comparacion_coef.plot(
    kind='bar',
    figsize=(13, 6)
)

plt.axhline(
    0,
    linewidth=1
)

plt.ylabel('Coeficiente sobre variables escaladas')
plt.title(
    'Comparación de coeficientes: '
    'Lineal, Ridge, Lasso y Elastic Net'
)
plt.xticks(rotation=70)
plt.tight_layout()
plt.show()

# 13. Resumen conceptual

| Modelo | Penalización | Efecto principal |
|---|---|---|
| Regresión lineal | Ninguna | Ajusta libremente los coeficientes |
| Ridge | L2: $b_j^2$ | Reduce coeficientes |
| Lasso | L1: $|b_j|$ | Puede llevar coeficientes a cero |
| Elastic Net | L1 + L2 | Combina reducción y selección |

## Tres ideas para recordar

### 1. `alpha` controla la fuerza de la regularización

$$
\alpha \uparrow
\Rightarrow
\text{mayor penalización}
$$

Pero una penalización excesiva puede producir underfitting.

### 2. Regularizar no garantiza mejorar

El desempeño debe comprobarse con datos que el modelo no utilizó para aprender.

### 3. Escalar es especialmente importante

Como la penalización depende del tamaño de los coeficientes, las variables deberían encontrarse en escalas comparables.

# 14. Mini reto

Sin modificar Train ni Validation:

1. prueba en Ridge:
   - `alpha=1`
   - `alpha=30`
   - `alpha=100`;

2. prueba en Lasso:
   - `alpha=0.01`
   - `alpha=0.05`
   - `alpha=0.30`;

3. prueba en Elastic Net:
   - `l1_ratio=0.2`
   - `l1_ratio=0.5`
   - `l1_ratio=0.8`;

4. responde:

> ¿Cuál obtiene el menor RMSE en Validation?

> ¿Cuál deja más coeficientes cercanos o iguales a cero?

> ¿Mejoró realmente respecto a la regresión lineal múltiple?

> ¿Qué ocurrió cuando la regularización fue demasiado fuerte?

No uses Test para escoger el ganador.

# 15. Puente hacia el Día 2

Todos los modelos de este notebook siguen teniendo una característica importante:

> Continúan construyendo una relación esencialmente **lineal** entre las variables y la predicción.

Ridge, Lasso y Elastic Net cambian **cómo se controlan los coeficientes**, pero no convierten por sí mismos la regresión en un modelo de árboles.

En el Día 2 cambiaremos completamente la forma de aprender.

En lugar de una ecuación como:

$$
\hat y=b_0+b_1x_1+\cdots+b_px_p
$$

estudiaremos modelos capaces de aprender reglas como:

```text
¿inversión <= cierto valor?
        /             \
      sí               no
     /                   \
¿inflación...?       ¿confianza...?
```

Ese será el punto de entrada a los **árboles de decisión**.